# Cargando los datos

In [41]:
import pandas as pd

#carcamos los dato de cada archivo csv en un DataFrame de pandas
df0 = pd.read_csv('geo_data_0.csv')
df1 = pd.read_csv('geo_data_1.csv')
df2 = pd.read_csv('geo_data_2.csv')

#mostramos la forma, información, descripción y las primeras filas de cada DataFrame
for df in [df0, df1, df2]:
    print(f"DataFrame shape: {df.shape}")
    print(f"DataFrame info:\n{df.info()}")
    print(f"DataFrame describe:\n{df.describe()}")
    print(f"DataFrame head:\n{df.head()}")
    print(f"Duplicate ID rows: {df['id'].duplicated().sum()}")
    print(f"Missing values:\n{df.isnull().sum()}")



DataFrame shape: (100000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
DataFrame info:
None
DataFrame describe:
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433       3.248248      44.288691
min        -1.408605      -0.848218     -12.088328       0.000000
25%        -0.072580      -0.200881       0.287748      56.497507
50%         0.502360       0.250252       2.515969      91.849972
75%         1.073581       0.700646       4.71508

Interpretamos las columnas f0, f1 y f2 como la calidad del petroleo y como caracteristicas para nuestro modelo y la columna product como la cantidad de reservas en miles de bariiles, este es nuestro objetivo.

# Modelo para predecir el volumen de reservas en pozos nuevos.

In [42]:
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

#Hagamos una funcion para entrenar y evaluar el modelo de regresión logística para cada DataFrame
def train_and_evaluate_model(df):
    #separamos las características (features) de la variable objetivo (target) y tambien separamos los id de los pozos para que no afecten al modelo
    features = df.drop(['product', 'id'], axis=1)
    target = df['product']
    
    #Dividimos los datos en conjuntos de entrenamiento y prueba
    features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.25, random_state=12345)

    #entrenamos el modelo de regresión logística
    model = LinearRegression()
    model.fit(features_train, target_train)

    #hacemos predicciones sobre el conjunto de prueba
    predictions = model.predict(features_test)

    #calculamos el error cuadrático medio, la raíz del error cuadrático medio, el volumen medio predicho y el volumen medio real
    rmse = np.sqrt(mean_squared_error(target_test, predictions))
    mean_pred = predictions.mean()
    mean_actual = target_test.mean()
    
    print(f"Volumen medio predicho: {mean_pred:.2f} miles de barriles")
    print(f"RMSE: {rmse:.2f}")
    print(f"Volumen medio real en validación: {mean_actual:.2f}")
    
    return model, predictions, target_test.reset_index(drop=True), rmse, mean_pred

print("Resultados geo_0")
model0, pred0, target_test0, rmse0, mean0 = train_and_evaluate_model(df0)
print("\nResultados geo_1")
model1, pred1, target_test1, rmse1, mean1 = train_and_evaluate_model(df1)
print("\nResultados geo_2")
model2, pred2, target_test2, rmse2, mean2 = train_and_evaluate_model(df2)


Resultados geo_0
Volumen medio predicho: 92.59 miles de barriles
RMSE: 37.58
Volumen medio real en validación: 92.08

Resultados geo_1
Volumen medio predicho: 68.73 miles de barriles
RMSE: 0.89
Volumen medio real en validación: 68.72

Resultados geo_2
Volumen medio predicho: 94.97 miles de barriles
RMSE: 40.03
Volumen medio real en validación: 94.88


# Pozos petrolíferos con los valores estimados más altos.

In [43]:
#Constantes del negocio
presupuesto = 100000000
barriles = 200
ganacia_por_barril = 4500

print(f"\nVolumen mínimo por pozo para no tener pérdidas: {presupuesto / (barriles * ganacia_por_barril):.2f} miles de barriles")


Volumen mínimo por pozo para no tener pérdidas: 111.11 miles de barriles


In [44]:
def calculate_profit(predictions, target_test, region_name):
    # Combinar predicciones con valores reales
    df_pred = pd.DataFrame({'predictions': predictions, 'real': target_test})
    
    # Seleccionar los 200 pozos con mayor predicción
    top_200 = df_pred.nlargest(200, 'predictions')
    
    # Ganancia real (usando el volumen REAL de esos pozos)
    total_volume = top_200['real'].sum()
    profit = total_volume * ganacia_por_barril - presupuesto
    
    print(f"\nRegión {region_name}:")
    print(f"Volumen total (200 mejores pozos): {total_volume:.2f} miles de barriles")
    print(f"Ganancia potencial: ${profit/1_000_000:.2f} millones de USD")
    
    return profit, total_volume

profit0, vol0 = calculate_profit(pred0, target_test0, "geo_0")
profit1, vol1 = calculate_profit(pred1, target_test1, "geo_1")
profit2, vol2 = calculate_profit(pred2, target_test2, "geo_2")


Región geo_0:
Volumen total (200 mejores pozos): 29601.84 miles de barriles
Ganancia potencial: $33.21 millones de USD

Región geo_1:
Volumen total (200 mejores pozos): 27589.08 miles de barriles
Ganancia potencial: $24.15 millones de USD

Región geo_2:
Volumen total (200 mejores pozos): 28245.22 miles de barriles
Ganancia potencial: $27.10 millones de USD


# Región con el beneficio total más alto para los pozos petrolíferos seleccionados.

In [45]:
def bootstrap_profit(predictions, target_test, region_name, n_samples=1000, wells=200):
    profits = []
    state = np.random.RandomState(12345)
    
    for i in range(n_samples):
        # Muestra bootstrap de los índices
        sample_indices = state.choice(len(predictions), size=len(predictions), replace=True)
        
        sample_pred = predictions[sample_indices]
        sample_real = target_test[sample_indices]
        
        # Seleccionar los 200 mejores según predicción
        df_sample = pd.DataFrame({'pred': sample_pred, 'real': sample_real})
        top_200 = df_sample.nlargest(wells, 'pred')
        
        revenue = top_200['real'].sum() * ganacia_por_barril
        profit = revenue - presupuesto
        profits.append(profit)
    
    profits = np.array(profits)
    
    mean_profit = profits.mean()
    lower = np.percentile(profits, 2.5)
    upper = np.percentile(profits, 97.5)
    risk_loss = (profits < 0).mean() * 100   # % de riesgo de pérdidas
    
    print(f"\nBootstrapping Región {region_name}:")
    print(f"Beneficio promedio: ${mean_profit/1_000_000:.2f} millones")
    print(f"Intervalo de confianza 95%: [${lower/1_000_000:.2f}, ${upper/1_000_000:.2f}] millones")
    print(f"Riesgo de pérdidas: {risk_loss:.2f}%")
    
    return mean_profit, lower, upper, risk_loss

# Ejecutar bootstrapping
print("\nANÁLISIS DE RIESGO CON BOOTSTRAPPING")
mean_p0, l0, u0, risk0 = bootstrap_profit(pred0, target_test0, region_name="geo_0")
mean_p1, l1, u1, risk1 = bootstrap_profit(pred1, target_test1, region_name="geo_1")
mean_p2, l2, u2, risk2 = bootstrap_profit(pred2, target_test2, region_name="geo_2")


ANÁLISIS DE RIESGO CON BOOTSTRAPPING

Bootstrapping Región geo_0:
Beneficio promedio: $33.16 millones
Intervalo de confianza 95%: [$29.86, $36.09] millones
Riesgo de pérdidas: 0.00%

Bootstrapping Región geo_1:
Beneficio promedio: $24.15 millones
Intervalo de confianza 95%: [$24.15, $24.15] millones
Riesgo de pérdidas: 0.00%

Bootstrapping Región geo_2:
Beneficio promedio: $27.12 millones
Intervalo de confianza 95%: [$23.48, $30.93] millones
Riesgo de pérdidas: 0.00%


La Región 1 tiene el valor de RMSE mas bajo lo que quiere decir que el modelo es más preciso y también más ajustado con la diferencia del intervalo de confianza = 0 , aunque tiene el volumen medio más bajo.
Todas las regiones cumplen con riesgo de pérdidas < 2.5%.
La región con mejor beneficio es la región 0 con $33.16 millones de dolares. 
